# 3. Cleaning and transformation

Run `python -m pip install -e ".[dev]"` from the repository root first. Each notebook runs independently and imports the same tested Python functions. Source timestamps retain their original local calendar meaning. Set `RETAIL_ETL_INPUT` to use another source and adjust `AS_OF` to its reporting cutoff.

In [ ]:
import os
from pathlib import Path
import pandas as pd
from retail_etl.pipeline import project_root, read_source
from retail_etl.transform import prepare_data, feature_engineering, validate_output

ROOT = project_root()
SOURCE = Path(os.getenv("RETAIL_ETL_INPUT", str(ROOT / "data/raw/Online Retail.xlsx")))
OUTPUT = Path(os.getenv("RETAIL_ETL_OUTPUT", str(ROOT / "outputs")))
AS_OF = "2011-12-09"  # Inclusive end date of the bundled historical dataset.


## Clean with the shared implementation

Identifiers remain strings. Text is trimmed, descriptions are uppercased, missing descriptions become UNKNOWN, dates are validated, and invalid rows are quarantined. No notebook-specific cleaning rules are maintained.

In [ ]:
raw = read_source(SOURCE)
prepared = prepare_data(raw, as_of=AS_OF)
cleaned = prepared.cleaned
cleaned.head()

## Inspect accepted types and coverage

In [ ]:
print(cleaned.dtypes)
print(f"Accepted {len(cleaned):,}; excluded {len(prepared.rejected):,}")

## Inspect unit price precision

A positive price such as £0.001 is retained. Decimal arithmetic avoids binary floating-point ambiguity. Line amounts are rounded half up to two decimal places; some legitimate sub-penny lines therefore have a zero rounded amount.

In [ ]:
final = feature_engineering(cleaned)
final.loc[final.total_amount.eq(0), ["invoice_no", "unit_price", "quantity", "total_amount"]].head()

## Validate before exporting

In [ ]:
validate_output(final)

The automation notebook or CLI owns exports, so this notebook cannot overwrite the final dataset with an incompatible schema.